# buy-open-sell-close

On the first trading day of each week, buy on the open, then sell on the close.

In [1]:
import pandas as pd
import datetime
import pinkfish as pf

# Format price data.
pd.options.display.float_format = '{:0.2f}'.format

Some global data

In [2]:
symbol = 'SPY'
capital = 10000
start = datetime.datetime(2015, 1, 1)
end = datetime.datetime.now()

Timeseries

In [3]:
# Fetch timeseries, select.
ts = pf.fetch_timeseries(symbol)
ts = pf.select_tradeperiod(ts, start, end, use_adj=False)

# Add calendar columns.
ts = pf.calendar(ts)

# Finalize timeseries.
ts, start = pf.finalize_timeseries(ts, start)

# Create tradelog and daily balance objects.
tlog = pf.TradeLog(symbol)
dbal = pf.DailyBal()

In [4]:
ts

,open,high,low,close,adj_close,volume,dotw,dotm,doty,month,first_dotw,first_dotm,first_doty,last_dotw,last_dotm,last_doty
date,,,,,,,,,,,,,,,,
2015-01-02,206.38,206.88,204.18,205.43,170.12,121465900,4,2,2,1,False,True,True,True,False,False
2015-01-05,204.17,204.37,201.35,201.72,167.05,169632600,0,5,5,1,True,False,False,False,False,False
2015-01-06,202.09,202.72,198.86,199.82,165.48,209151400,1,6,6,1,False,False,False,False,False,False
2015-01-07,201.42,202.72,200.88,202.31,167.54,125346700,2,7,7,1,False,False,False,False,False,False
2015-01-08,204.01,206.16,203.99,205.90,170.51,147217800,3,8,8,1,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-08,743.36,745.34,738.19,739.22,739.22,49319100,0,8,159,6,True,False,False,False,False,False
2026-06-09,743.63,746.90,722.59,737.05,737.05,87683500,1,9,160,6,False,False,False,False,False,False
2026-06-10,733.39,738.38,725.33,725.43,725.43,60341300,2,10,161,6,False,False,False,False,False,False


Algorithm

In [5]:
pf.TradeLog.cash = capital

# Loop through timeseries.
for i, row in enumerate(ts.itertuples()):

    date = row.Index.to_pydatetime()
    end_flag = pf.is_last_row(ts, i)

    # On first trading day of the week, buy open, then sell close.
    if row.first_dotw:
        tlog.buy(date, row.open)
        tlog.sell(date, row.close)

    # Record daily balance.
    dbal.append(date, row.close, row.high, row.low)

Retrieve logs

In [6]:
tlog = tlog.get_log()
dbal = dbal.get_log(tlog)

In [7]:
tlog.tail()

,entry_date,entry_price,exit_date,exit_price,pl_points,pl_cash,qty,cumul_total,direction,symbol
592,2026-05-11,736.45,2026-05-11,739.30,2.85,51.30,18,3596.45,LONG,SPY
593,2026-05-18,739.83,2026-05-18,738.65,-1.18,-21.24,18,3575.21,LONG,SPY
594,2026-05-26,750.01,2026-05-26,750.59,0.58,10.44,18,3585.65,LONG,SPY
595,2026-06-01,755.36,2026-06-01,758.54,3.18,54.06,17,3639.71,LONG,SPY
596,2026-06-08,743.36,2026-06-08,739.22,-4.14,-74.52,18,3565.19,LONG,SPY


Get stats

In [8]:
stats = pf.stats(ts, tlog, dbal, capital)

Summary

In [9]:
pf.summary(stats)

,strategy
annual_return_rate,2.70
max_closed_out_drawdown,-13.98
best_month,6.17
worst_month,-5.08
sharpe_ratio,0.54
sortino_ratio,0.33
monthly_std,1.41
annual_std,6.11
